# 05 — Full workflow_update pipeline

**NON_BASELINE_RUN**. Reuses current Data by default; pass `--run-data` only when a fresh network fetch is intended.

In [ ]:
import subprocess
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "CLAUDE.md").exists():
    ROOT = ROOT.parent
BASE = [
    "--config",
    "configs/base.yaml",
    "--profile",
    "configs/profiles/workflow_update.yaml",
    "--override",
    "configs/provisional/workflow_update_downstream.yaml",
]


def run(name, cmd):
    print("$", " ".join(cmd), flush=True)
    t = time.perf_counter()
    p = subprocess.run(cmd, cwd=ROOT, check=False)
    elapsed = time.perf_counter() - t
    print(f"[{name}] exit={p.returncode} elapsed={elapsed:.2f}s")
    if p.returncode:
        raise RuntimeError(f"{name} failed")
    return elapsed

In [ ]:
total_s = run(
    "workflow-update",
    [
        "uv",
        "run",
        "qshield-pipeline",
        "workflow-update",
        *BASE,
        "--quantum-mode",
        "exact",
    ],
)

In [ ]:
import json

p = ROOT / "artifacts/dev/risk/final_recommendation.json"
x = json.loads(p.read_text())
print("actual_solver", x.get("actual_solver"), "materiality", x.get("materiality_met"))
print("CVaR", x.get("true_cvar_before"), "->", x.get("true_cvar_after"))